In [1]:
import torch
from pyqcu import tools, dslash, lattice
from pyqcu.cuda import qcu, define
from pyqcu.cuda.define import params, argv, set_ptrs
params[define._LAT_X_] = 4
params[define._LAT_Y_] = 4
params[define._LAT_Z_] = 4
params[define._LAT_T_] = 8
params[define._LAT_XYZT_] = params[define._LAT_X_] * \
    params[define._LAT_Y_]*params[define._LAT_Z_]*params[define._LAT_T_]
params[define._GRID_X_], params[define._GRID_Y_], params[define._GRID_Z_], params[
    define._GRID_T_] = tools.give_grid_size()
params[define._PARITY_] = 0
params[define._NODE_RANK_] = define.rank
params[define._NODE_SIZE_] = define.size
params[define._DAGGER_] = 0
params[define._MAX_ITER_] = 1000
params[define._DATA_TYPE_] = define._LAT_C64_
# params[define._DATA_TYPE_] = define._LAT_C128_
params[define._SET_INDEX_] = 0
params[define._SET_PLAN_] = 1
params[define._MG_X_] = 1
params[define._MG_Y_] = 1
params[define._MG_Z_] = 1
params[define._MG_T_] = 1
params[define._LAT_E_] = 24
params[define._VERBOSE_] = 1
params[define._SEED_] = 42
argv = argv.to(dtype=define.dtype(params[define._DATA_TYPE_]).to_real())
argv[define._MASS_] = 0.05
argv[define._TOL_] = 1e-9
argv[define._SIGMA_] = 0.1
print(params)
print(argv)
print(set_ptrs)
gauge_eo = torch.zeros(size=[2, 3, 3, 4]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                       params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
fermion_in_eo = torch.zeros(size=[2, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                            params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
fermion_in_eo = torch.rand_like(fermion_in_eo)
fermion_out_eo = torch.zeros(size=[2, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                             params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([   4,    4,    4,    8,  512,    1,    1,    1,    1,    0,    0,    1,
           0, 1000,    2,    0,    1,    1,    1,    1,    1,   24,    1,   42,
           0], dtype=torch.int32)
tensor([5.0000e-02, 1.0000e-09, 1.0000e-01])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [2]:
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] = 0
params[define._SET_PLAN_] = -1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyGaussGaugeQcu(gauge_eo, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
print(lattice.check_su3(U=gauge_eo[0]))
print(lattice.check_su3(U=gauge_eo[1]))

set_ptr:0x55819692bdf0
set_ptrs:0x5581955cc400
long long set_ptr:94015065341424
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :0
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :0
host_params[_SET_PLAN_]      :-1
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
hos

In [3]:
_gauge_eo=gauge_eo.clone()
_fermion_in_eo=fermion_in_eo.clone()
_fermion_out_eo=fermion_out_eo.clone()

In [4]:
print('Difference:', tools.norm(gauge_eo-_gauge_eo)/tools.norm(_gauge_eo))
print('Difference:', tools.norm(fermion_in_eo-_fermion_in_eo)/tools.norm(_fermion_in_eo))

Difference: 0.0
Difference: 0.0


In [ ]:

print(set_ptrs)
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyWilsonBistabCgQcu(
    fermion_out_eo, fermion_in_eo, gauge_eo, set_ptrs, params)
# qcu.applyEndQcu(set_ptrs, params)
print(set_ptrs)
qcu_dest = tools.poooxyzt2oooxyzt(input_array=fermion_out_eo)
qcu_U = tools.poooxyzt2oooxyzt(input_array=gauge_eo)
qcu_src = tools.poooxyzt2oooxyzt(input_array=fermion_in_eo)
refer_src = dslash.give_wilson(
    src=qcu_dest, U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8), with_I=True)
print('qcu_src:', qcu_src.flatten()[:100])
print('refer_src:', refer_src.flatten()[:100])
print('Difference:', tools.norm(refer_src-qcu_src)/tools.norm(qcu_src))


NameError: name 'set_ptrs' is not defined

In [6]:

clover_ee = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                           params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_ee_inv = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                               params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_oo = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                           params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_oo_inv = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                               params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
# gauge_eo = torch.ones_like(gauge_eo)*2
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 2
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloversQcu(clover_ee, clover_ee_inv, gauge_eo, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
print(set_ptrs)
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 2
params[define._PARITY_] = 1
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloversQcu(clover_oo, clover_oo_inv, gauge_eo, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
print(set_ptrs)
fermion_out_eo = torch.zeros_like(fermion_out_eo)
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloverBistabCgQcu(fermion_out_eo, fermion_in_eo, gauge_eo,
                           clover_ee, clover_oo, clover_ee_inv, clover_oo_inv,  set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
print(set_ptrs)
qcu_dest = tools.poooxyzt2oooxyzt(input_array=fermion_out_eo)
qcu_U = tools.poooxyzt2oooxyzt(input_array=gauge_eo)
qcu_src = tools.poooxyzt2oooxyzt(input_array=fermion_in_eo)
refer_clover_term = dslash.make_clover(
    U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8))
refer_src = dslash.give_wilson(
    src=qcu_dest, U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8), with_I=True)+dslash.give_clover(src=qcu_dest, clover_term=refer_clover_term)
print('qcu_src:', qcu_src.flatten()[:100])
print('refer_src:', refer_src.flatten()[:100])
print('Difference:', tools.norm(refer_src-qcu_src)/tools.norm(qcu_src))

set_ptr:0x5581a3df6330
set_ptrs:0x5581955cc400
long long set_ptr:94015288468272
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :0
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :2
host_params[_SET_PLAN_]      :2
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
host

In [7]:

clover_eeoo = torch.zeros(size=[2, 4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                                params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_eeoo[0] = clover_ee
clover_eeoo[1] = clover_oo
qcu_clover_term = tools.poooxyzt2oooxyzt(
    input_array=clover_eeoo)
qcu_clover_term = dslash.cut_I(clover_term=qcu_clover_term)

qcu_clover_term_eo = tools.oooxyzt2poooxyzt(
    input_array=qcu_clover_term)
refer_clover_term_eo = tools.oooxyzt2poooxyzt(
    input_array=refer_clover_term)
print('qcu_clover_term:', qcu_clover_term.flatten()[:100])
print('refer_clover_term:', refer_clover_term.flatten()[:100])
# print('qcu_clover_term_eo:', qcu_clover_term_eo[0].flatten()[:100])
# print('refer_clover_term_eo:', refer_clover_term_eo[0].flatten()[:100])
# print('qcu_clover_term_eo:', qcu_clover_term_eo[1].flatten()[:100])
# print('refer_clover_term_eo:', refer_clover_term_eo[1].flatten()[:100])
print('Difference:', tools.norm(refer_clover_term -
      qcu_clover_term)/tools.norm(qcu_clover_term))


qcu_clover_term: tensor([-3.8316e-03+0.j, -1.8643e-03+0.j,  3.2039e-03+0.j,  5.3358e-04+0.j,
        -1.7998e-03+0.j, -5.4159e-03+0.j,  2.0691e-03+0.j,  4.6600e-03+0.j,
        -5.0867e-03+0.j, -3.2208e-03+0.j,  9.8002e-04+0.j,  1.9778e-03+0.j,
         5.4185e-03+0.j,  3.1829e-05+0.j, -6.9264e-03+0.j, -5.8151e-03+0.j,
        -1.9433e-03+0.j,  3.4924e-03+0.j, -8.2582e-04+0.j, -2.0717e-03+0.j,
         6.3369e-03+0.j, -3.9542e-04+0.j, -1.4666e-03+0.j,  3.9876e-04+0.j,
         2.7074e-03+0.j, -1.4642e-03+0.j,  6.1376e-03+0.j,  2.6786e-04+0.j,
        -2.5557e-03+0.j,  3.4189e-04+0.j,  2.9922e-05+0.j,  7.3220e-03+0.j,
        -8.8704e-04+0.j,  2.8539e-04+0.j,  2.8368e-03+0.j, -2.4422e-03+0.j,
         1.9132e-03+0.j,  3.7313e-05+0.j, -2.8713e-03+0.j,  8.3513e-03+0.j,
         2.1443e-03+0.j, -3.1715e-03+0.j,  2.2817e-04+0.j, -1.7223e-03+0.j,
         4.2026e-03+0.j,  2.7715e-03+0.j, -1.9099e-03+0.j,  2.6815e-03+0.j,
        -3.4442e-03+0.j,  5.8651e-05+0.j,  3.4399e-03+0.j,  3.8651e-03+

In [8]:
print('qcu_dest:', qcu_dest.flatten()[:100])

qcu_dest: tensor([14.3822+4.2503j, 13.4864+5.0078j, 13.4088+4.3973j, 14.1719+4.6220j,
        14.5201+4.4026j, 14.1554+4.6038j, 13.6687+4.0887j, 13.4937+4.3030j,
        13.8027+5.3743j, 13.2941+4.2487j, 13.3982+4.6728j, 13.6875+4.2480j,
        15.3262+3.9597j, 13.6589+4.3323j, 14.3942+4.1293j, 13.9882+4.3828j,
        14.6232+4.3989j, 14.4268+4.8981j, 14.0892+4.7600j, 13.9053+4.1953j,
        14.3208+4.3580j, 14.3086+5.6930j, 14.2091+3.9841j, 13.0847+3.9443j,
        13.6708+3.7253j, 13.8975+3.9150j, 13.3414+3.3048j, 13.2826+3.7839j,
        13.9495+3.1610j, 14.3668+4.3842j, 13.0816+3.7261j, 12.7513+4.0453j,
        15.0266+3.7633j, 13.6905+4.4177j, 13.2282+4.2106j, 14.0351+4.1814j,
        13.6756+4.3569j, 13.4445+4.3482j, 13.7329+4.2051j, 13.6131+4.1311j,
        14.2605+4.2857j, 14.1324+4.2113j, 12.4749+4.5015j, 13.7013+3.7707j,
        14.3664+3.4059j, 13.6877+4.2013j, 14.4272+4.5924j, 13.4845+4.5949j,
        14.7877+5.0538j, 14.8436+4.6408j, 13.8648+4.3794j, 14.2209+4.8948j,
  

In [9]:
a = qcu_src-refer_src
print('a:', a.flatten()[:100])

a: tensor([-4.7684e-07-1.0133e-06j, -5.4017e-06-1.2517e-06j,
         0.0000e+00-3.5763e-07j,  4.4703e-06+1.6838e-06j,
         2.3246e-06-1.0729e-06j,  3.0994e-06-3.8743e-07j,
         1.0133e-06+3.4273e-07j,  7.3165e-06-2.8014e-06j,
        -1.4305e-06+1.2517e-06j,  4.3213e-07-5.9605e-07j,
         4.9472e-06-1.3113e-06j,  2.2352e-07-9.3877e-07j,
         2.5630e-06+8.9407e-07j,  5.0664e-07+7.7486e-07j,
         8.3447e-07+3.6657e-06j,  4.1723e-07-6.9290e-07j,
        -5.9605e-08-4.4703e-07j,  2.7418e-06+2.8014e-06j,
        -5.9605e-08-1.1921e-07j,  3.9339e-06+6.8545e-07j,
         1.3113e-06-4.7684e-07j, -2.6822e-07-2.1458e-06j,
        -7.1526e-07+5.9605e-07j,  2.9430e-06-6.7055e-07j,
         2.4550e-06-4.1723e-07j, -4.7684e-07-3.2037e-07j,
        -1.7285e-06+5.3644e-07j, -8.3447e-07+8.5682e-07j,
         3.2187e-06+2.1607e-06j,  2.0266e-06-5.3644e-07j,
        -3.6228e-07-2.8014e-06j,  3.3528e-08+1.3113e-06j,
         7.1526e-07-4.0829e-06j, -3.3081e-06-5.9605e-08j,
         4.

In [10]:
qcu_src

tensor([[[[[[5.1426e-01+2.8105e-01j, 1.1031e-01+9.6042e-01j,
             5.1488e-01+8.8442e-01j,  ...,
             9.0473e-01+3.9067e-01j, 8.0917e-01+2.4857e-01j,
             1.3617e-01+8.0154e-01j],
            [4.2073e-01+9.9429e-01j, 2.2695e-01+7.6475e-01j,
             6.1631e-01+9.9533e-01j,  ...,
             2.6500e-01+6.1439e-01j, 8.4649e-01+1.9230e-01j,
             7.1293e-01+3.2191e-02j],
            [9.0423e-01+3.3300e-01j, 6.3895e-01+9.8769e-01j,
             8.9603e-01+1.7233e-01j,  ...,
             1.1530e-01+7.6721e-01j, 6.3448e-01+8.0517e-01j,
             7.3475e-02+1.8649e-01j],
            [5.2526e-02+2.9118e-01j, 6.5636e-01+1.0970e-01j,
             4.9432e-01+2.4659e-01j,  ...,
             5.4467e-01+3.6956e-01j, 1.1908e-02+3.6358e-01j,
             2.4041e-02+6.6447e-01j]],

           [[8.6271e-01+4.5402e-01j, 3.6502e-01+6.5304e-01j,
             6.5244e-01+1.7324e-01j,  ...,
             5.3528e-01+5.4297e-01j, 3.2214e-01+4.7920e-01j,
             1.2268e-

In [11]:
refer_src

tensor([[[[[[5.1426e-01+2.8105e-01j, 1.1031e-01+9.6042e-01j,
             5.1488e-01+8.8442e-01j,  ...,
             9.0473e-01+3.9068e-01j, 8.0917e-01+2.4857e-01j,
             1.3616e-01+8.0155e-01j],
            [4.2073e-01+9.9428e-01j, 2.2695e-01+7.6475e-01j,
             6.1630e-01+9.9533e-01j,  ...,
             2.6500e-01+6.1439e-01j, 8.4649e-01+1.9230e-01j,
             7.1293e-01+3.2191e-02j],
            [9.0423e-01+3.3300e-01j, 6.3895e-01+9.8769e-01j,
             8.9603e-01+1.7233e-01j,  ...,
             1.1530e-01+7.6721e-01j, 6.3449e-01+8.0517e-01j,
             7.3472e-02+1.8650e-01j],
            [5.2524e-02+2.9118e-01j, 6.5636e-01+1.0970e-01j,
             4.9432e-01+2.4659e-01j,  ...,
             5.4467e-01+3.6956e-01j, 1.1909e-02+3.6358e-01j,
             2.4041e-02+6.6447e-01j]],

           [[8.6271e-01+4.5402e-01j, 3.6502e-01+6.5304e-01j,
             6.5244e-01+1.7324e-01j,  ...,
             5.3528e-01+5.4297e-01j, 3.2215e-01+4.7920e-01j,
             1.2268e-

In [12]:
fermion_out_eo = torch.zeros_like(fermion_out_eo)
fermion_in_o=fermion_in_eo[1]
fermion_out_e=fermion_out_eo[0]
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 2
params[define._PARITY_] = 1
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloverDslashQcu(fermion_out_e, fermion_in_o, gauge_eo, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
refer_clover_term = dslash.make_clover(
    U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8))
refer_clover_term = dslash.add_I(clover_term=refer_clover_term)
refer_clover_term_inv = dslash.inverse(clover_term=refer_clover_term)
refer_clover_term_inv_eo = tools.oooxyzt2poooxyzt(
    input_array=refer_clover_term_inv)
refer_clover_term_inv_e = refer_clover_term_inv_eo[0]
refer_fermion_out_e = dslash.give_clover_ee(src_e=dslash.give_wilson_eo(
    src_o=fermion_in_o, U_eo=gauge_eo, kappa=1 / (2 * argv[define._MASS_] + 8)),clover_e=refer_clover_term_inv_e)
print(tools.norm(clover_ee_inv-refer_clover_term_inv_e)/tools.norm(clover_ee_inv))
print(tools.norm(fermion_out_e-refer_fermion_out_e)/tools.norm(fermion_out_e))

set_ptr:0x5581a5914890
set_ptrs:0x5581955cc400
long long set_ptr:94015316904080
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :1
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :5
host_params[_SET_PLAN_]      :2
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
host

In [13]:
fermion_out_eo = torch.zeros_like(fermion_out_eo)
fermion_in_o=fermion_in_eo[1]
fermion_out_e=fermion_out_eo[0]
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 1
params[define._PARITY_] = 1
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyWilsonDslashQcu(fermion_out_e, fermion_in_o, gauge_eo, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
refer_fermion_out_e = dslash.give_wilson_eo(
    src_o=fermion_in_o, U_eo=gauge_eo, kappa=1 / (2 * argv[define._MASS_] + 8))
print(fermion_out_e.flatten()[:100])
print(refer_fermion_out_e.flatten()[:100])
print(tools.norm(fermion_out_e-refer_fermion_out_e)/tools.norm(fermion_out_e))

set_ptr:0x5581a70e04b0
set_ptrs:0x5581955cc400
long long set_ptr:94015341855920
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :1
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :6
host_params[_SET_PLAN_]      :1
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
host

In [14]:
print("gauge_eo.is_contiguous():",gauge_eo.is_contiguous())
print("fermion_in_eo.is_contiguous():",fermion_in_eo.is_contiguous())
print("fermion_in_out.is_contiguous():",fermion_out_eo.is_contiguous())
print("qcu_src.is_contiguous():",qcu_src.is_contiguous())
print("qcu_dest.is_contiguous():",qcu_dest.is_contiguous())

gauge_eo.is_contiguous(): True
fermion_in_eo.is_contiguous(): True
fermion_in_out.is_contiguous(): True
qcu_src.is_contiguous(): True
qcu_dest.is_contiguous(): True
